In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import display
import ipywidgets as widgets
from sklearn.cluster import DBSCAN

# --------------------------------------------------
# Load data
# --------------------------------------------------
csv_path = r"/Users/maxladabaum/Downloads/1_merged_channel_merge_x_step15nm.csv"
df = pd.read_csv(csv_path, usecols=["x", "y"])

x = df["x"].to_numpy()
y = df["y"].to_numpy()

good = np.isfinite(x) & np.isfinite(y)
x = x[good]
y = y[good]

x_min_global, x_max_global = float(np.min(x)), float(np.max(x))
y_min_global, y_max_global = float(np.min(y)), float(np.max(y))

print(f"Loaded {len(x):,} localizations")

# --------------------------------------------------
# Spatial subsampling helper
# --------------------------------------------------
def spatial_subsample_mask(x_vals, y_vals, step):
    if step <= 1:
        return np.ones_like(x_vals, dtype=bool)

    x0 = x_vals.min()
    y0 = y_vals.min()

    xb = np.floor((x_vals - x0) / step).astype(np.int64)
    yb = np.floor((y_vals - y0) / step).astype(np.int64)

    return (xb % step == 0) & (yb % step == 0)

# --------------------------------------------------
# Helper to create slider + text box pair
# --------------------------------------------------
def make_linked_float(name, val, vmin, vmax, step):
    slider = widgets.FloatSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.FloatText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

def make_linked_int(name, val, vmin, vmax, step=1):
    slider = widgets.IntSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.IntText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

# --------------------------------------------------
# Plot + cluster
# --------------------------------------------------
def plot_clusters(xmin, xmax, ymin, ymax, spatial_step, bins, cluster_radius, min_samples):
    # Crop
    mask_crop = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    x_crop = x[mask_crop]
    y_crop = y[mask_crop]

    if len(x_crop) == 0:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_title("No localizations in selected region")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    # Spatial subsample
    mask_sub = spatial_subsample_mask(x_crop, y_crop, max(1, int(spatial_step)))
    x_plot = x_crop[mask_sub]
    y_plot = y_crop[mask_sub]

    if len(x_plot) == 0:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_title("No localizations after subsampling")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    pts = np.column_stack([x_plot, y_plot])

    # DBSCAN clustering
    clustering = DBSCAN(eps=float(cluster_radius), min_samples=int(min_samples))
    labels = clustering.fit_predict(pts)

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    # Plot density image
    fig, ax = plt.subplots(figsize=(8, 8))
    h = ax.hist2d(
        x_plot,
        y_plot,
        bins=int(bins),
        range=[[xmin, xmax], [ymin, ymax]]
    )
    plt.colorbar(h[3], ax=ax, label="Counts per bin")

    # Draw red circles around clusters
    for lab in unique_labels:
        cluster_pts = pts[labels == lab]
        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()

        # radius enclosing cluster
        dists = np.sqrt((cluster_pts[:, 0] - cx)**2 + (cluster_pts[:, 1] - cy)**2)
        r = dists.max() if len(dists) else cluster_radius

        circle = Circle(
            (cx, cy),
            r,
            edgecolor="red",
            facecolor="none",
            linewidth=2
        )
        ax.add_patch(circle)

    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(
        f"Displayed localizations: {len(x_plot):,} | "
        f"Clusters found: {len(unique_labels)} | "
        f"Noise points: {(labels == -1).sum():,}"
    )
    plt.show()

# --------------------------------------------------
# Controls
# --------------------------------------------------
x_step = max((x_max_global - x_min_global) / 500, 1e-6)
y_step = max((y_max_global - y_min_global) / 500, 1e-6)

xmin_box, xmin_slider = make_linked_float("x min", x_min_global, x_min_global, x_max_global, x_step)
xmax_box, xmax_slider = make_linked_float("x max", x_max_global, x_min_global, x_max_global, x_step)
ymin_box, ymin_slider = make_linked_float("y min", y_min_global, y_min_global, y_max_global, y_step)
ymax_box, ymax_slider = make_linked_float("y max", y_max_global, y_min_global, y_max_global, y_step)

spatial_box, spatial_slider = make_linked_int("spatial step", 4, 1, 50)
bins_box, bins_slider = make_linked_int("bins", 250, 50, 800, 25)

# You may need to adjust these depending on your x/y units
radius_box, radius_slider = make_linked_float("cluster radius", 30.0, 1.0, 500.0, 1.0)
minsamp_box, minsamp_slider = make_linked_int("min samples", 10, 1, 200)

ui = widgets.VBox([
    xmin_box,
    xmax_box,
    ymin_box,
    ymax_box,
    spatial_box,
    bins_box,
    radius_box,
    minsamp_box
])

out = widgets.interactive_output(
    plot_clusters,
    {
        "xmin": xmin_slider,
        "xmax": xmax_slider,
        "ymin": ymin_slider,
        "ymax": ymax_slider,
        "spatial_step": spatial_slider,
        "bins": bins_slider,
        "cluster_radius": radius_slider,
        "min_samples": minsamp_slider,
    }
)

display(ui, out)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import display
import ipywidgets as widgets
from sklearn.cluster import DBSCAN

# --------------------------------------------------
# Load data
# --------------------------------------------------
csv_path = r"/Users/maxladabaum/Downloads/1_merged_channel_merge_x_step15nm.csv"
df = pd.read_csv(csv_path, usecols=["x", "y"])

x = df["x"].to_numpy()
y = df["y"].to_numpy()

good = np.isfinite(x) & np.isfinite(y)
x = x[good]
y = y[good]

x_min_global, x_max_global = float(np.min(x)), float(np.max(x))
y_min_global, y_max_global = float(np.min(y)), float(np.max(y))

print(f"Loaded {len(x):,} localizations")

# --------------------------------------------------
# Spatial subsampling helper
# --------------------------------------------------
def spatial_subsample_mask(x_vals, y_vals, step):
    if step <= 1:
        return np.ones_like(x_vals, dtype=bool)

    x0 = x_vals.min()
    y0 = y_vals.min()

    xb = np.floor((x_vals - x0) / step).astype(np.int64)
    yb = np.floor((y_vals - y0) / step).astype(np.int64)

    return (xb % step == 0) & (yb % step == 0)

# --------------------------------------------------
# Helper to create slider + text box pair
# --------------------------------------------------
def make_linked_float(name, val, vmin, vmax, step):
    slider = widgets.FloatSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.FloatText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

def make_linked_int(name, val, vmin, vmax, step=1):
    slider = widgets.IntSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.IntText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

# --------------------------------------------------
# Main plotting function
# --------------------------------------------------
def plot_clusters_and_overlay(
    xmin, xmax, ymin, ymax,
    spatial_step, bins,
    cluster_radius, min_samples,
    overlay_alpha, overlay_point_size,
    min_cluster_size_for_overlay
):
    # Crop to chosen field of view
    mask_crop = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    x_crop = x[mask_crop]
    y_crop = y[mask_crop]

    if len(x_crop) == 0:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_title("No localizations in selected region")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    # Spatial subsample for speed
    mask_sub = spatial_subsample_mask(x_crop, y_crop, max(1, int(spatial_step)))
    x_plot = x_crop[mask_sub]
    y_plot = y_crop[mask_sub]

    if len(x_plot) == 0:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_title("No localizations after subsampling")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    pts = np.column_stack([x_plot, y_plot])

    # Radius-based clustering
    clustering = DBSCAN(eps=float(cluster_radius), min_samples=int(min_samples))
    labels = clustering.fit_predict(pts)

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    # --------------------------------------------------
    # Plot 1: field of view with red circles
    # --------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    ax0, ax1 = axes

    h = ax0.hist2d(
        x_plot,
        y_plot,
        bins=int(bins),
        range=[[xmin, xmax], [ymin, ymax]]
    )
    plt.colorbar(h[3], ax=ax0, label="Counts per bin")

    cluster_sizes = {}
    for lab in unique_labels:
        cluster_pts = pts[labels == lab]
        cluster_sizes[lab] = len(cluster_pts)

        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()
        dists = np.sqrt((cluster_pts[:, 0] - cx)**2 + (cluster_pts[:, 1] - cy)**2)
        r = dists.max() if len(dists) else cluster_radius

        circle = Circle(
            (cx, cy),
            r,
            edgecolor="red",
            facecolor="none",
            linewidth=2
        )
        ax0.add_patch(circle)

    ax0.set_aspect("equal")
    ax0.set_xlabel("x")
    ax0.set_ylabel("y")
    ax0.set_title(
        f"Field of view\n"
        f"Displayed localizations: {len(x_plot):,} | "
        f"Clusters: {len(unique_labels)} | "
        f"Noise: {(labels == -1).sum():,}"
    )

    # --------------------------------------------------
    # Plot 2: all clusters overlaid after centering
    # --------------------------------------------------
    cmap = plt.cm.get_cmap("tab20", max(len(unique_labels), 1))
    overlay_count = 0
    max_extent = 0.0

    for i, lab in enumerate(unique_labels):
        cluster_pts = pts[labels == lab]

        if len(cluster_pts) < int(min_cluster_size_for_overlay):
            continue

        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()

        # center cluster on its centroid
        x_centered = cluster_pts[:, 0] - cx
        y_centered = cluster_pts[:, 1] - cy

        ax1.scatter(
            x_centered,
            y_centered,
            s=float(overlay_point_size),
            alpha=float(overlay_alpha),
            color=cmap(i),
            edgecolors="none"
        )

        max_extent = max(
            max_extent,
            np.max(np.abs(x_centered)) if len(x_centered) else 0,
            np.max(np.abs(y_centered)) if len(y_centered) else 0
        )
        overlay_count += 1

    ax1.set_aspect("equal")
    ax1.set_xlabel("x relative to cluster centroid")
    ax1.set_ylabel("y relative to cluster centroid")
    ax1.set_title(
        f"Centered overlay of clusters\n"
        f"Clusters shown: {overlay_count}"
    )

    if overlay_count > 0 and max_extent > 0:
        lim = max_extent * 1.1
        ax1.set_xlim(-lim, lim)
        ax1.set_ylim(-lim, lim)

    plt.tight_layout()
    plt.show()

# --------------------------------------------------
# Controls
# --------------------------------------------------
x_step = max((x_max_global - x_min_global) / 500, 1e-6)
y_step = max((y_max_global - y_min_global) / 500, 1e-6)

xmin_box, xmin_slider = make_linked_float("x min", x_min_global, x_min_global, x_max_global, x_step)
xmax_box, xmax_slider = make_linked_float("x max", x_max_global, x_min_global, x_max_global, x_step)
ymin_box, ymin_slider = make_linked_float("y min", y_min_global, y_min_global, y_max_global, y_step)
ymax_box, ymax_slider = make_linked_float("y max", y_max_global, y_min_global, y_max_global, y_step)

spatial_box, spatial_slider = make_linked_int("spatial step", 4, 1, 50)
bins_box, bins_slider = make_linked_int("bins", 250, 50, 800, 25)

radius_box, radius_slider = make_linked_float("cluster radius", 30.0, 1.0, 500.0, 1.0)
minsamp_box, minsamp_slider = make_linked_int("min samples", 10, 1, 200)

alpha_box, alpha_slider = make_linked_float("overlay alpha", 0.15, 0.01, 1.0, 0.01)
psize_box, psize_slider = make_linked_float("point size", 5.0, 0.1, 20.0, 0.1)
minc_box, minc_slider = make_linked_int("min cluster size", 10, 1, 500)

ui = widgets.VBox([
    xmin_box,
    xmax_box,
    ymin_box,
    ymax_box,
    spatial_box,
    bins_box,
    radius_box,
    minsamp_box,
    alpha_box,
    psize_box,
    minc_box
])

out = widgets.interactive_output(
    plot_clusters_and_overlay,
    {
        "xmin": xmin_slider,
        "xmax": xmax_slider,
        "ymin": ymin_slider,
        "ymax": ymax_slider,
        "spatial_step": spatial_slider,
        "bins": bins_slider,
        "cluster_radius": radius_slider,
        "min_samples": minsamp_slider,
        "overlay_alpha": alpha_slider,
        "overlay_point_size": psize_slider,
        "min_cluster_size_for_overlay": minc_slider,
    }
)

display(ui, out)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import display
import ipywidgets as widgets
from sklearn.cluster import DBSCAN

# --------------------------------------------------
# Load data
# --------------------------------------------------
csv_path = r"/Users/maxladabaum/Downloads/1_merged_channel_merge_x_step15nm.csv"
df = pd.read_csv(csv_path, usecols=["x", "y"])

x = df["x"].to_numpy()
y = df["y"].to_numpy()

good = np.isfinite(x) & np.isfinite(y)
x = x[good]
y = y[good]

x_min_global, x_max_global = float(np.min(x)), float(np.max(x))
y_min_global, y_max_global = float(np.min(y)), float(np.max(y))

print(f"Loaded {len(x):,} localizations")

# --------------------------------------------------
# Spatial subsampling helper
# --------------------------------------------------
def spatial_subsample_mask(x_vals, y_vals, step):
    if step <= 1:
        return np.ones_like(x_vals, dtype=bool)

    x0 = x_vals.min()
    y0 = y_vals.min()

    xb = np.floor((x_vals - x0) / step).astype(np.int64)
    yb = np.floor((y_vals - y0) / step).astype(np.int64)

    return (xb % step == 0) & (yb % step == 0)

# --------------------------------------------------
# Widget helpers
# --------------------------------------------------
def make_linked_float(name, val, vmin, vmax, step):
    slider = widgets.FloatSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.FloatText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

def make_linked_int(name, val, vmin, vmax, step=1):
    slider = widgets.IntSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.IntText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

# --------------------------------------------------
# Main plotting function
# --------------------------------------------------
def plot_clusters_overlay_and_histograms(
    xmin, xmax, ymin, ymax,
    spatial_step, bins,
    cluster_radius, min_samples,
    overlay_alpha, overlay_point_size,
    min_cluster_size_for_overlay,
    hist_bins
):
    # Crop to FOV
    mask_crop = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    x_crop = x[mask_crop]
    y_crop = y[mask_crop]

    if len(x_crop) == 0:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_title("No localizations in selected region")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    # Spatial subsample for speed
    mask_sub = spatial_subsample_mask(x_crop, y_crop, max(1, int(spatial_step)))
    x_plot = x_crop[mask_sub]
    y_plot = y_crop[mask_sub]

    if len(x_plot) == 0:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_title("No localizations after subsampling")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    pts = np.column_stack([x_plot, y_plot])

    # DBSCAN clustering
    clustering = DBSCAN(eps=float(cluster_radius), min_samples=int(min_samples))
    labels = clustering.fit_predict(pts)

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    # Keep only clustered points for histograms
    clustered_mask = labels != -1
    pts_clustered = pts[clustered_mask]

    # --------------------------------------------------
    # Create 2x2 figure
    # --------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    ax0, ax1, ax2, ax3 = axes.flatten()

    # --------------------------------------------------
    # Plot 1: field of view with red circles
    # --------------------------------------------------
    h = ax0.hist2d(
        x_plot,
        y_plot,
        bins=int(bins),
        range=[[xmin, xmax], [ymin, ymax]]
    )
    plt.colorbar(h[3], ax=ax0, label="Counts per bin")

    for lab in unique_labels:
        cluster_pts = pts[labels == lab]
        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()

        dists = np.sqrt((cluster_pts[:, 0] - cx)**2 + (cluster_pts[:, 1] - cy)**2)
        r = dists.max() if len(dists) else cluster_radius

        circle = Circle(
            (cx, cy),
            r,
            edgecolor="red",
            facecolor="none",
            linewidth=2
        )
        ax0.add_patch(circle)

    ax0.set_aspect("equal")
    ax0.set_xlabel("x")
    ax0.set_ylabel("y")
    ax0.set_title(
        f"Field of view\n"
        f"Displayed localizations: {len(x_plot):,} | "
        f"Clusters: {len(unique_labels)} | "
        f"Noise: {(labels == -1).sum():,}"
    )

    # --------------------------------------------------
    # Plot 2: centered overlay of clusters
    # --------------------------------------------------
    cmap = plt.cm.get_cmap("tab20", max(len(unique_labels), 1))
    overlay_count = 0
    max_extent = 0.0

    for i, lab in enumerate(unique_labels):
        cluster_pts = pts[labels == lab]

        if len(cluster_pts) < int(min_cluster_size_for_overlay):
            continue

        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()

        x_centered = cluster_pts[:, 0] - cx
        y_centered = cluster_pts[:, 1] - cy

        ax1.scatter(
            x_centered,
            y_centered,
            s=float(overlay_point_size),
            alpha=float(overlay_alpha),
            color=cmap(i),
            edgecolors="none"
        )

        max_extent = max(
            max_extent,
            np.max(np.abs(x_centered)) if len(x_centered) else 0,
            np.max(np.abs(y_centered)) if len(y_centered) else 0
        )
        overlay_count += 1

    ax1.set_aspect("equal")
    ax1.set_xlabel("x relative to cluster centroid")
    ax1.set_ylabel("y relative to cluster centroid")
    ax1.set_title(f"Centered overlay of clusters\nClusters shown: {overlay_count}")

    if overlay_count > 0 and max_extent > 0:
        lim = max_extent * 1.1
        ax1.set_xlim(-lim, lim)
        ax1.set_ylim(-lim, lim)

    # --------------------------------------------------
    # Plot 3: histogram of clustered localizations vs x
    # --------------------------------------------------
    if len(pts_clustered) > 0:
        ax2.hist(pts_clustered[:, 0], bins=int(hist_bins), range=(xmin, xmax))
    ax2.set_xlabel("x")
    ax2.set_ylabel("Localization count")
    ax2.set_title("Clustered localizations only: counts vs x")

    # --------------------------------------------------
    # Plot 4: histogram of clustered localizations vs y
    # --------------------------------------------------
    if len(pts_clustered) > 0:
        ax3.hist(pts_clustered[:, 1], bins=int(hist_bins), range=(ymin, ymax))
    ax3.set_xlabel("y")
    ax3.set_ylabel("Localization count")
    ax3.set_title("Clustered localizations only: counts vs y")

    plt.tight_layout()
    plt.show()

# --------------------------------------------------
# Controls
# --------------------------------------------------
x_step = max((x_max_global - x_min_global) / 500, 1e-6)
y_step = max((y_max_global - y_min_global) / 500, 1e-6)

xmin_box, xmin_slider = make_linked_float("x min", x_min_global, x_min_global, x_max_global, x_step)
xmax_box, xmax_slider = make_linked_float("x max", x_max_global, x_min_global, x_max_global, x_step)
ymin_box, ymin_slider = make_linked_float("y min", y_min_global, y_min_global, y_max_global, y_step)
ymax_box, ymax_slider = make_linked_float("y max", y_max_global, y_min_global, y_max_global, y_step)

spatial_box, spatial_slider = make_linked_int("spatial step", 4, 1, 50)
bins_box, bins_slider = make_linked_int("bins", 250, 50, 800, 25)

radius_box, radius_slider = make_linked_float("cluster radius", 30.0, 1.0, 500.0, 1.0)
minsamp_box, minsamp_slider = make_linked_int("min samples", 10, 1, 200)

alpha_box, alpha_slider = make_linked_float("overlay alpha", 0.15, 0.01, 1.0, 0.01)
psize_box, psize_slider = make_linked_float("point size", 5.0, 0.1, 20.0, 0.1)
minc_box, minc_slider = make_linked_int("min cluster size", 10, 1, 500)

histbins_box, histbins_slider = make_linked_int("hist bins", 80, 10, 500, 5)

ui = widgets.VBox([
    xmin_box,
    xmax_box,
    ymin_box,
    ymax_box,
    spatial_box,
    bins_box,
    radius_box,
    minsamp_box,
    alpha_box,
    psize_box,
    minc_box,
    histbins_box
])

out = widgets.interactive_output(
    plot_clusters_overlay_and_histograms,
    {
        "xmin": xmin_slider,
        "xmax": xmax_slider,
        "ymin": ymin_slider,
        "ymax": ymax_slider,
        "spatial_step": spatial_slider,
        "bins": bins_slider,
        "cluster_radius": radius_slider,
        "min_samples": minsamp_slider,
        "overlay_alpha": alpha_slider,
        "overlay_point_size": psize_slider,
        "min_cluster_size_for_overlay": minc_slider,
        "hist_bins": histbins_slider
    }
)

display(ui, out)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import display
import ipywidgets as widgets
from sklearn.cluster import DBSCAN

# --------------------------------------------------
# Load data
# --------------------------------------------------
csv_path = r"/Users/maxladabaum/Downloads/1_merged_channel_merge_x_step15nm.csv"
df = pd.read_csv(csv_path, usecols=["x", "y"])

x = df["x"].to_numpy()
y = df["y"].to_numpy()

good = np.isfinite(x) & np.isfinite(y)
x = x[good]
y = y[good]

x_min_global, x_max_global = float(np.min(x)), float(np.max(x))
y_min_global, y_max_global = float(np.min(y)), float(np.max(y))

print(f"Loaded {len(x):,} localizations")

# --------------------------------------------------
# Spatial subsampling helper
# --------------------------------------------------
def spatial_subsample_mask(x_vals, y_vals, step):
    if step <= 1:
        return np.ones_like(x_vals, dtype=bool)

    x0 = x_vals.min()
    y0 = y_vals.min()

    xb = np.floor((x_vals - x0) / step).astype(np.int64)
    yb = np.floor((y_vals - y0) / step).astype(np.int64)

    return (xb % step == 0) & (yb % step == 0)

# --------------------------------------------------
# Widget helpers
# --------------------------------------------------
def make_linked_float(name, val, vmin, vmax, step):
    slider = widgets.FloatSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.FloatText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

def make_linked_int(name, val, vmin, vmax, step=1):
    slider = widgets.IntSlider(
        value=val,
        min=vmin,
        max=vmax,
        step=step,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="500px")
    )
    text = widgets.IntText(
        value=val,
        layout=widgets.Layout(width="140px")
    )
    widgets.jslink((slider, "value"), (text, "value"))
    return widgets.HBox([slider, text]), slider

# --------------------------------------------------
# Main plotting function
# --------------------------------------------------
def plot_clusters_overlay_and_aligned_histograms(
    xmin, xmax, ymin, ymax,
    spatial_step, bins,
    cluster_radius, min_samples,
    overlay_alpha, overlay_point_size,
    min_cluster_size_for_overlay,
    hist_bins
):
    # Crop to FOV
    mask_crop = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    x_crop = x[mask_crop]
    y_crop = y[mask_crop]

    if len(x_crop) == 0:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_title("No localizations in selected region")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    # Spatial subsample for speed
    mask_sub = spatial_subsample_mask(x_crop, y_crop, max(1, int(spatial_step)))
    x_plot = x_crop[mask_sub]
    y_plot = y_crop[mask_sub]

    if len(x_plot) == 0:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_title("No localizations after subsampling")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.show()
        return

    pts = np.column_stack([x_plot, y_plot])

    # DBSCAN clustering
    clustering = DBSCAN(eps=float(cluster_radius), min_samples=int(min_samples))
    labels = clustering.fit_predict(pts)
    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    # --------------------------------------------------
    # Build aligned cluster pool
    # --------------------------------------------------
    aligned_x_all = []
    aligned_y_all = []

    cluster_sizes = {}
    kept_labels_for_overlay = []

    for lab in unique_labels:
        cluster_pts = pts[labels == lab]
        cluster_sizes[lab] = len(cluster_pts)

        if len(cluster_pts) < int(min_cluster_size_for_overlay):
            continue

        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()

        x_centered = cluster_pts[:, 0] - cx
        y_centered = cluster_pts[:, 1] - cy

        aligned_x_all.append(x_centered)
        aligned_y_all.append(y_centered)
        kept_labels_for_overlay.append(lab)

    if len(aligned_x_all) > 0:
        aligned_x_all = np.concatenate(aligned_x_all)
        aligned_y_all = np.concatenate(aligned_y_all)
    else:
        aligned_x_all = np.array([])
        aligned_y_all = np.array([])

    # --------------------------------------------------
    # Make plots
    # --------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    ax0, ax1, ax2, ax3 = axes.flatten()

    # --------------------------------------------------
    # Plot 1: FOV with red circles
    # --------------------------------------------------
    h = ax0.hist2d(
        x_plot,
        y_plot,
        bins=int(bins),
        range=[[xmin, xmax], [ymin, ymax]]
    )
    plt.colorbar(h[3], ax=ax0, label="Counts per bin")

    for lab in unique_labels:
        cluster_pts = pts[labels == lab]
        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()
        dists = np.sqrt((cluster_pts[:, 0] - cx)**2 + (cluster_pts[:, 1] - cy)**2)
        r = dists.max() if len(dists) else cluster_radius

        circle = Circle(
            (cx, cy),
            r,
            edgecolor="red",
            facecolor="none",
            linewidth=2
        )
        ax0.add_patch(circle)

    ax0.set_aspect("equal")
    ax0.set_xlabel("x")
    ax0.set_ylabel("y")
    ax0.set_title(
        f"Field of view\n"
        f"Displayed localizations: {len(x_plot):,} | "
        f"Clusters: {len(unique_labels)} | "
        f"Noise: {(labels == -1).sum():,}"
    )

    # --------------------------------------------------
    # Plot 2: centered overlay of clusters
    # --------------------------------------------------
    cmap = plt.cm.get_cmap("tab20", max(len(kept_labels_for_overlay), 1))
    overlay_count = 0
    max_extent = 0.0

    for i, lab in enumerate(kept_labels_for_overlay):
        cluster_pts = pts[labels == lab]
        cx = cluster_pts[:, 0].mean()
        cy = cluster_pts[:, 1].mean()

        x_centered = cluster_pts[:, 0] - cx
        y_centered = cluster_pts[:, 1] - cy

        ax1.scatter(
            x_centered,
            y_centered,
            s=float(overlay_point_size),
            alpha=float(overlay_alpha),
            color=cmap(i),
            edgecolors="none"
        )

        if len(x_centered) > 0:
            max_extent = max(
                max_extent,
                np.max(np.abs(x_centered)),
                np.max(np.abs(y_centered))
            )
        overlay_count += 1

    ax1.set_aspect("equal")
    ax1.set_xlabel("x relative to cluster centroid")
    ax1.set_ylabel("y relative to cluster centroid")
    ax1.set_title(f"Centered overlay of clusters\nClusters shown: {overlay_count}")

    if overlay_count > 0 and max_extent > 0:
        lim = max_extent * 1.1
        ax1.set_xlim(-lim, lim)
        ax1.set_ylim(-lim, lim)
    else:
        lim = 1.0

    # --------------------------------------------------
    # Plot 3: histogram of aligned x
    # --------------------------------------------------
    if len(aligned_x_all) > 0:
        ax2.hist(aligned_x_all, bins=int(hist_bins), range=(-lim, lim))
    ax2.set_xlabel("x relative to cluster centroid")
    ax2.set_ylabel("Localization count")
    ax2.set_title("Aligned clusters: counts vs relative x")

    # --------------------------------------------------
    # Plot 4: histogram of aligned y
    # --------------------------------------------------
    if len(aligned_y_all) > 0:
        ax3.hist(aligned_y_all, bins=int(hist_bins), range=(-lim, lim))
    ax3.set_xlabel("y relative to cluster centroid")
    ax3.set_ylabel("Localization count")
    ax3.set_title("Aligned clusters: counts vs relative y")

    plt.tight_layout()
    plt.show()

# --------------------------------------------------
# Controls
# --------------------------------------------------
x_step = max((x_max_global - x_min_global) / 500, 1e-6)
y_step = max((y_max_global - y_min_global) / 500, 1e-6)

xmin_box, xmin_slider = make_linked_float("x min", x_min_global, x_min_global, x_max_global, x_step)
xmax_box, xmax_slider = make_linked_float("x max", x_max_global, x_min_global, x_max_global, x_step)
ymin_box, ymin_slider = make_linked_float("y min", y_min_global, y_min_global, y_max_global, y_step)
ymax_box, ymax_slider = make_linked_float("y max", y_max_global, y_min_global, y_max_global, y_step)

spatial_box, spatial_slider = make_linked_int("spatial step", 4, 1, 50)
bins_box, bins_slider = make_linked_int("bins", 250, 50, 800, 25)

radius_box, radius_slider = make_linked_float("cluster radius", 30.0, 1.0, 500.0, 1.0)
minsamp_box, minsamp_slider = make_linked_int("min samples", 10, 1, 200)

alpha_box, alpha_slider = make_linked_float("overlay alpha", 0.15, 0.01, 1.0, 0.01)
psize_box, psize_slider = make_linked_float("point size", 5.0, 0.1, 20.0, 0.1)
minc_box, minc_slider = make_linked_int("min cluster size", 10, 1, 500)

histbins_box, histbins_slider = make_linked_int("hist bins", 80, 10, 500, 5)

ui = widgets.VBox([
    xmin_box,
    xmax_box,
    ymin_box,
    ymax_box,
    spatial_box,
    bins_box,
    radius_box,
    minsamp_box,
    alpha_box,
    psize_box,
    minc_box,
    histbins_box
])

out = widgets.interactive_output(
    plot_clusters_overlay_and_aligned_histograms,
    {
        "xmin": xmin_slider,
        "xmax": xmax_slider,
        "ymin": ymin_slider,
        "ymax": ymax_slider,
        "spatial_step": spatial_slider,
        "bins": bins_slider,
        "cluster_radius": radius_slider,
        "min_samples": minsamp_slider,
        "overlay_alpha": alpha_slider,
        "overlay_point_size": psize_slider,
        "min_cluster_size_for_overlay": minc_slider,
        "hist_bins": histbins_slider
    }
)

display(ui, out)